# Healthcare Claims Fraud Detection

## Provider-Level Exploratory Data Analysis

This notebook analyzes provider-level healthcare utilization, reimbursement patterns, beneficiary characteristics, and fraud behavior using the engineered feature dataset.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
pd.set_option("display.max_columns", None)
print("Libraries Imported Successfully")

Libraries Imported Successfully


## Load Provider-Level Feature Dataset

In [2]:
provider_df = pd.read_csv("../data/processed/provider_features.csv")
print("Dataset Loaded Successfully")

Dataset Loaded Successfully


## Dataset Overview

In [3]:
print("Shapep:",provider_df.shape)
provider_df.head()

Shapep: (5410, 20)


,Provider,PotentialFraud,InpatientClaimCount,UniqueInpatientBeneficiaries,AvgInpatientReimbursement,TotalInpatientReimbursement,AvgLengthOfStay,MaxLengthOfStay,AvgDiagnosisCount,OutpatientClaimCount,UniqueOutpatientBeneficiaries,AvgOutpatientReimbursement,TotalOutpatientReimbursement,AvgPatientAge,AvgChronicConditions,DeceasedRate,AvgIPAnnualReimbursement,AvgOPAnnualReimbursement,UniqueBeneficiaries,Fraud
0,PRV51001,No,5.0,5.0,19400.000000,97000.0,5.000000,14.0,7.200000,20.0,19.0,382.000000,7640.0,78.125000,5.541667,0.000000,18047.916667,2537.500000,24,0
1,PRV51003,Yes,62.0,53.0,9241.935484,573000.0,5.161290,27.0,8.112903,70.0,66.0,466.714286,32670.0,68.957265,4.367521,0.008547,6814.017094,2490.598291,117,1
2,PRV51004,No,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,149.0,138.0,350.134228,52170.0,72.485507,4.318841,0.007246,4596.739130,2095.144928,138,0
3,PRV51005,Yes,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,1165.0,495.0,241.124464,280910.0,69.987879,3.884848,0.006061,3717.232323,1798.808081,495,1
4,PRV51007,No,3.0,3.0,6333.333333,19000.0,5.333333,7.0,7.333333,69.0,56.0,213.188406,14710.0,67.982759,3.879310,0.017241,3109.655172,1497.241379,58,0


In [4]:
provider_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5410 entries, 0 to 5409
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Provider                       5410 non-null   str    
 1   PotentialFraud                 5410 non-null   str    
 2   InpatientClaimCount            5410 non-null   float64
 3   UniqueInpatientBeneficiaries   5410 non-null   float64
 4   AvgInpatientReimbursement      5410 non-null   float64
 5   TotalInpatientReimbursement    5410 non-null   float64
 6   AvgLengthOfStay                5410 non-null   float64
 7   MaxLengthOfStay                5410 non-null   float64
 8   AvgDiagnosisCount              5410 non-null   float64
 9   OutpatientClaimCount           5410 non-null   float64
 10  UniqueOutpatientBeneficiaries  5410 non-null   float64
 11  AvgOutpatientReimbursement     5410 non-null   float64
 12  TotalOutpatientReimbursement   5410 non-null   float64
 13 

## Fraud Distribution Analysis

In [5]:
provider_df["Fraud"].value_counts()

Fraud
0    4904
1     506
Name: count, dtype: int64

In [6]:
round(provider_df["Fraud"] .value_counts(normalize=True) * 100, 2)

Fraud
0    90.65
1     9.35
Name: proportion, dtype: float64

In [7]:
fraud_counts = (provider_df["Fraud"].value_counts().reset_index())
fraud_counts.columns = ["Fraud","Count"]
fraud_counts["Fraud"] = (
    fraud_counts["Fraud"]
    .map({
        0: "Non-Fraud",
        1: "Fraud"
    })
)
fig = px.pie(
    fraud_counts,
    names="Fraud",
    values="Count",
    hole=0.5,
    title="Provider Fraud Distribution"
)
fig.update_layout(title_x=0.5)
fig.show()

### Observation

The dataset is moderately imbalanced, with non-fraudulent providers accounting for the majority of observations.

Fraudulent providers represent approximately 9% of all providers, indicating that fraud detection is a minority-class classification problem. This class imbalance must be considered during model development and evaluation.

## Fraud vs Non-Fraud Provider Comparison

In [8]:
fraud_comparison = (provider_df.groupby("Fraud").mean(numeric_only=True).T)
fraud_comparison.columns = ["NonFraud", "Fraud"]
fraud_comparison

,NonFraud,Fraud
InpatientClaimCount,3.481240,46.249012
UniqueInpatientBeneficiaries,3.261215,40.756917
AvgInpatientReimbursement,3363.579498,9605.429866
TotalInpatientReimbursement,34055.568923,476854.762846
AvgLengthOfStay,1.868263,5.316086
MaxLengthOfStay,4.476550,23.750988
AvgDiagnosisCount,2.726830,7.048478
OutpatientClaimCount,66.954119,374.296443
UniqueOutpatientBeneficiaries,46.124592,206.770751
AvgOutpatientReimbursement,262.215760,473.618176


### Key Findings

Fraudulent providers exhibit significantly higher healthcare utilization and reimbursement activity compared to non-fraudulent providers.

Key observations:

- Fraudulent providers process approximately 13 times more inpatient claims.
- Fraudulent providers serve substantially more beneficiaries in both inpatient and outpatient settings.
- Average inpatient reimbursement is nearly 3 times higher among fraudulent providers.
- Fraudulent providers have longer patient stays and higher diagnosis counts.
- Total inpatient and outpatient reimbursements are dramatically higher for fraudulent providers.
- Patient demographics such as age and chronic condition burden are relatively similar across both groups.

These findings suggest that provider behavior and claim volume are stronger fraud indicators than patient demographics.

## Top Features Differentiating Fraudulent Providers

In [9]:
comparison_plot = fraud_comparison.copy()
comparison_plot["Difference"] = (comparison_plot["Fraud"] - comparison_plot["NonFraud"])
comparison_plot = (comparison_plot.sort_values("Difference", ascending=False).head(10))
fig = px.bar(
    comparison_plot.reset_index(),
    x="Difference",
    y="index",
    orientation="h",
    title="Top Features Associated with Fraudulent Providers",
    text_auto=".2s"
)
fig.update_layout(
    title_x=0.5,
    yaxis_title="Feature",
    xaxis_title="Fraud - NonFraud Difference"
)
fig.show()

### Observation

Reimbursement-related variables are the strongest differentiators between fraudulent and non-fraudulent providers.

Fraudulent providers generate substantially higher inpatient and outpatient reimbursements while also serving a much larger number of beneficiaries. The findings indicate that claim volume and reimbursement behavior are more predictive of fraud than patient demographic characteristics.

## Correlation Analysis

In [11]:
corr_matrix = provider_df.drop(columns=["Provider", "PotentialFraud"]).corr()
fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    title="Provider Feature Correlation Heatmap"
)
fig.update_layout(
    title_x=0.5,
    width=1200,
    height=900
)
fig.show()

### Correlation Insights

Several strong correlations are observed among provider-level features.

Key findings:

- InpatientClaimCount and TotalInpatientReimbursement show an almost perfect positive correlation.
- OutpatientClaimCount, UniqueOutpatientBeneficiaries, and TotalOutpatientReimbursement are highly correlated, indicating that claim volume strongly influences reimbursement totals.
- Length of Stay and Diagnosis Count exhibit strong positive relationships with reimbursement measures.
- Patient demographic variables such as Age and Deceased Rate show very weak correlations with Fraud.
- Fraud is most strongly associated with:
  - MaxLengthOfStay
  - InpatientClaimCount
  - TotalInpatientReimbursement
  - UniqueInpatientBeneficiaries
  - UniqueBeneficiaries

These findings suggest that provider utilization behavior and reimbursement activity are stronger fraud indicators than patient demographics.

## Feature Correlation with Fraud

In [12]:
fraud_corr = (corr_matrix["Fraud"].drop("Fraud").sort_values(ascending=False))
fraud_corr

MaxLengthOfStay                  0.547667
TotalInpatientReimbursement      0.532795
InpatientClaimCount              0.525393
UniqueInpatientBeneficiaries     0.522256
UniqueBeneficiaries              0.393531
UniqueOutpatientBeneficiaries    0.340550
TotalOutpatientReimbursement     0.337756
OutpatientClaimCount             0.335803
AvgDiagnosisCount                0.315956
AvgLengthOfStay                  0.308969
AvgInpatientReimbursement        0.307556
AvgIPAnnualReimbursement         0.127398
AvgChronicConditions             0.065790
AvgOutpatientReimbursement       0.046770
DeceasedRate                     0.012353
AvgPatientAge                    0.001162
AvgOPAnnualReimbursement        -0.015204
Name: Fraud, dtype: float64

### Fraud Correlation Ranking

The strongest predictors of provider fraud are primarily related to healthcare utilization and reimbursement behavior.

Top fraud-associated features include:

- Maximum Length of Stay
- Total Inpatient Reimbursement
- Inpatient Claim Count
- Unique Inpatient Beneficiaries
- Unique Beneficiaries

Patient demographic characteristics such as Age and Deceased Rate show little relationship with fraud, indicating that provider behavior is a much stronger indicator than patient characteristics.

In [13]:
fraud_corr_df = (fraud_corr.reset_index())
fraud_corr_df.columns = ["Feature", "Correlation"]
fig = px.bar(
    fraud_corr_df,
    x="Correlation",
    y="Feature",
    orientation="h",
    title="Feature Correlation with Fraud",
    text_auto=".2f"
)
fig.update_layout(
    title_x=0.5,
    height=700
)
fig.show()

## Distribution Analysis of Key Fraud Features

In [14]:
fig = px.box(
    provider_df,
    x="PotentialFraud",
    y="TotalInpatientReimbursement",
    color="PotentialFraud",
    title="Fraud vs Non-Fraud: Total Inpatient Reimbursement"
)
fig.update_layout(title_x=0.5)
fig.show()

### Observation

Fraudulent providers receive significantly higher total inpatient reimbursements than non-fraudulent providers.

The median reimbursement for fraudulent providers is substantially larger, and the presence of numerous extreme outliers indicates unusually high billing volumes.

This supports the hypothesis that abnormal reimbursement behavior is a major indicator of healthcare fraud.

## Fraud vs Non-Fraud: Unique Beneficiaries

In [15]:
fig = px.box(
    provider_df,
    x="PotentialFraud",
    y="UniqueBeneficiaries",
    color="PotentialFraud",
    title="Fraud vs Non-Fraud: Unique Beneficiaries"
)
fig.update_layout(title_x=0.5)
fig.show()

### Observation

Fraudulent providers serve substantially more beneficiaries than non-fraudulent providers.

The median number of beneficiaries for fraudulent providers is considerably higher, and several providers exhibit extremely large beneficiary counts. This indicates that fraudulent providers generally operate at a larger scale and submit claims for a significantly broader patient population.

Combined with higher reimbursement amounts, beneficiary volume appears to be an important indicator of potential healthcare fraud.

## Fraud vs Non-Fraud: Inpatient Claim Count

In [16]:
fig = px.box(
    provider_df,
    x="PotentialFraud",
    y="InpatientClaimCount",
    color="PotentialFraud",
    title="Fraud vs Non-Fraud: Inpatient Claim Count"
)
fig.update_layout(title_x=0.5)
fig.show()

### Observation

Fraudulent providers exhibit significantly higher inpatient claim volumes compared to non-fraudulent providers.

The median inpatient claim count for fraudulent providers is many times larger than that of legitimate providers. Additionally, fraudulent providers display a wider distribution and more extreme outliers, indicating unusually high claim activity.

This further confirms that claim volume is one of the strongest indicators of provider fraud.

# Save Final EDA Dataset

In [17]:
provider_df.to_csv("../data/processed/provider_fraud_modeling_dataset.csv",index=False)
print("Dataset Saved Successfully")

Dataset Saved Successfully


# Key Findings Summary

## Exploratory Data Analysis (EDA) Conclusion

The exploratory analysis revealed clear behavioral differences between fraudulent and non-fraudulent healthcare providers.

### Key Insights

- Fraudulent providers process significantly more inpatient and outpatient claims.
- Fraudulent providers serve substantially larger beneficiary populations.
- Fraudulent providers receive much higher total reimbursement amounts.
- Inpatient-related metrics are stronger indicators of fraud than demographic features.
- Length of stay, inpatient claim count, and total inpatient reimbursement showed the strongest positive relationship with fraud.
- Patient age and mortality rate demonstrated little to no relationship with fraudulent behavior.

### Most Important Fraud Indicators

1. MaxLengthOfStay
2. TotalInpatientReimbursement
3. InpatientClaimCount
4. UniqueInpatientBeneficiaries
5. UniqueBeneficiaries
6. TotalOutpatientReimbursement
7. OutpatientClaimCount

These findings provide strong evidence that provider billing behavior and claim volume patterns are the primary drivers of healthcare fraud detection.